# 02c - Radio Telemetry Preprocessing V2 (Member 2, Phase 1)

Implements the improved radio preprocessing pipeline from the technical audit. All logic lives in
`radio_preprocessing_v2.py` (pandas + NumPy only); this notebook runs it **step by step**, shows the evidence at each stage,
writes the V2 outputs and verifies that nothing pre-existing was touched.

**Rules honoured:** `radio_clean.csv` and every pre-existing repository file are read-only (verified by SHA-256 at the end) ·
no labels are used to clean/flag/aggregate · raw values are never altered (problems become flags) · missing values are never imputed ·
counter resets are **not** clipped to zero · no network-radio alignment and no timezone conversion of radio timestamps here.

**Outputs (new files only):** `radio_window_features_v2.csv`, `radio_feature_columns_v2.json`, `radio_quality_report_v2.md`,
`radio_duplicate_removal_log_v2.csv`, `radio_counter_diagnostics_v2.csv`.

Run from the `Radio Dataset/` folder (or the repo root): *Kernel -> Restart & Run All*, or
`jupyter nbconvert --to notebook --execute --inplace "02c-radio-preprocessing-v2.ipynb"`.

In [1]:
import sys, hashlib, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

sys.dont_write_bytecode = True            # do not create __pycache__ inside the repository

HERE = Path.cwd()
if not (HERE / "radio_preprocessing_v2.py").exists():
    HERE = HERE / "Radio Dataset"
assert (HERE / "radio_preprocessing_v2.py").exists(), "Run from the 'Radio Dataset' folder or the repo root"
REPO = HERE.parent
sys.path.insert(0, str(HERE))
import radio_preprocessing_v2 as rp

INPUT, OUT = HERE / "radio_clean.csv", HERE
NEW_FILES = {rp.OUT_WINDOW, rp.OUT_JSON, rp.OUT_REPORT, rp.OUT_DUPLOG, rp.OUT_COUNTERS, rp.OUT_CONFLICTS,
             "02c-radio-preprocessing-v2.ipynb", "radio_preprocessing_v2.py"}

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

# Manifest of EVERY pre-existing repository file (taken before anything is written)
PRE_EXISTING = sorted(p for p in REPO.rglob("*")
                      if p.is_file() and p.name not in NEW_FILES
                      and not {"__pycache__", ".git", ".ipynb_checkpoints"} & set(p.parts))
MANIFEST_BEFORE = {str(p.relative_to(REPO)): sha(p) for p in PRE_EXISTING}

print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | {rp.PREPROCESSING_VERSION}")
print(f"input : {INPUT}")
print(f"output: {OUT}")
print(f"pre-existing repository files hashed: {len(MANIFEST_BEFORE)}")
assert rp.DEFAULT_INPUT.resolve() == INPUT.resolve()

python 3.12.3 | pandas 3.0.6 | numpy 2.5.4 | radio_preprocessing_v2.0.0
input : /home/claude/repo/Radio Dataset/radio_clean.csv
output: /home/claude/repo/Radio Dataset
pre-existing repository files hashed: 17


## Step 1 - Data integrity: schema, types, scenarios, missing values, chronological sort, duplicates

In [2]:
p = rp.RadioPipeline(INPUT, OUT)
p.step1_integrity()

print(f"loaded {len(p.raw):,} rows x {p.raw.shape[1]} columns | input SHA-256 {p.input_sha[:16]}...")
display(p.checks.frame()[["passed", "name", "detail"]])

loaded 7,490 rows x 53 columns | input SHA-256 67aa11be8f8c9c9d...


,passed,name,detail
0,True,schema: all required columns present,53 required columns present
1,True,schema: no unexpected extra columns (informati...,none
2,True,types: numeric columns are numeric dtype,non-numeric=[]
3,True,types: no +/-inf values in numeric columns,none
4,True,types: key/time columns are integer dtype,not integer=[]
5,True,"types: no nulls in key columns (scenario, ue_i...","{'scenario': 0, 'ue_id': 0, 'timestamp_ms': 0}"
6,True,scenarios: scenario-level metadata constant wi...,35 scenarios
7,True,scenarios: every scenario has at least one UE,min UEs=1
8,True,"types: rnti_int equals int(rnti, 16) for every...",
9,True,duplicates: no conflicting measurement values ...,"35 duplicate groups, all payload-identical"


In [3]:
print("Missing values (columns with any):")
display(p.missing[p.missing["missing"] > 0] if (p.missing["missing"] > 0).any() else "no missing values in any input column")

print("\nRows per scenario (in / duplicates removed / kept):")
display(p.rows_per_scenario.T)

Missing values (columns with any):


'no missing values in any input column'


Rows per scenario (in / duplicates removed / kept):


scenario,Benign1,Benign2,Benign3,Benign4,Benign5,Benign6,DDOS_ICMP1,DDOS_ICMP2,DDOS_ICMP3,DDOS_SYN1,...,SYN2,SYN3,TCP1,TCP2,TCP3,UDP1,UDP2,UDP3,UDP4,UDP5
rows_in,257,585,607,611,313,609,243,231,229,213,...,125,124,125,124,124,126,125,125,119,127
duplicates_removed,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
rows_kept,256,584,606,610,312,608,242,230,228,212,...,124,123,124,123,123,125,124,124,118,126


In [4]:
d = p.dup
print(f"duplicate key (scenario, ue_id, timestamp_ms): {d['n_rows_in_duplicate_groups']} rows in {d['n_duplicate_groups']} groups "
      f"-> {d['n_extra_copies']} extra copies")
print(f"groups with CONFLICTING measurement payload: {len(d['conflicts'])}   (a conflict would stop the pipeline)")
lg = p.removal_log
print(f"removed {len(lg)} records | payload identical: {int(lg['measurement_payload_identical'].sum())} | "
      f"all non-provenance columns identical: {int(lg['all_non_provenance_columns_identical'].sum())} | "
      f"first sample of its UE series: {int(lg['is_first_sample_of_ue_series'].sum())} | source: {lg['source_fmt'].value_counts().to_dict()}")
display(lg.head(6))

duplicate key (scenario, ue_id, timestamp_ms): 70 rows in 35 groups -> 35 extra copies
groups with CONFLICTING measurement payload: 0   (a conflict would stop the pipeline)
removed 35 records | payload identical: 35 | all non-provenance columns identical: 35 | first sample of its UE series: 35 | source: {'csv': 33, 'json': 2}


,scenario,ue_id,timestamp_ms,timestamp_s,removed_file_row,kept_file_row,folder,source_fmt,file_order_desc,input_is_duplicate_flag,rnti,n_rows_in_key_group,measurement_payload_identical,all_non_provenance_columns_identical,is_first_sample_of_ue_series,reason
0,Benign1,1,1744634077522,1744634077,256,255,Benign Traffic,csv,True,1,d52d,2,True,True,True,exact duplicate of earlier record with identic...
1,Benign2,2,1744634413522,1744634413,584,583,Benign Traffic,csv,True,1,a271,2,True,True,True,exact duplicate of earlier record with identic...
2,Benign3,2,1744635027522,1744635027,606,605,Benign Traffic,csv,True,1,a271,2,True,True,True,exact duplicate of earlier record with identic...
3,Benign4,2,1744635550522,1744635550,610,609,Benign Traffic,csv,True,1,a271,2,True,True,True,exact duplicate of earlier record with identic...
4,Benign5,8,1744636112522,1744636112,312,311,Benign Traffic,csv,True,1,dbef,2,True,True,True,exact duplicate of earlier record with identic...
5,Benign6,2,1744636438522,1744636438,608,607,Benign Traffic,csv,True,1,a271,2,True,True,True,exact duplicate of earlier record with identic...


## Step 2 - Timestamp validation (epoch milliseconds are used as-is; no timezone conversion)

In [5]:
p.step2_timestamps()
ts_checks = p.checks.frame()
display(ts_checks[ts_checks["name"].str.startswith("timestamps")][["passed", "name", "detail"]])
pt = p.stats["time"]["per_scenario_time"][["start_utc", "end_utc"]].copy()
pt["duration_s"] = (pt["end_utc"] - pt["start_utc"]).dt.total_seconds()
display(pt)

,passed,name,detail
10,True,timestamps: timestamp_ms is integer and positive,
11,True,timestamps: every timestamp_ms has 13 digits (...,digit counts={13: 7490}
12,True,timestamps: all values decode to a UTC date in...,decoded range 2025-02-27 13:39:07.404000+00:00...
13,True,timestamps: timestamp_s == floor(timestamp_ms ...,mismatches=0
14,True,timestamps: time_utc column equals the UTC dec...,mismatches=0
15,True,timestamps: time_utc strings carry an explicit...,
16,True,timestamps: radio_window_id == scenario@timest...,
17,True,timestamps: strictly increasing within every (...,non-positive steps=0


,start_utc,end_utc,duration_s
scenario,,,
Benign1,2025-04-14 12:34:37.522000+00:00,2025-04-14 12:36:44.522000+00:00,127.000
Benign2,2025-04-14 12:40:13.522000+00:00,2025-04-14 12:45:31.522000+00:00,318.000
Benign3,2025-04-14 12:50:27.522000+00:00,2025-04-14 12:55:29.522000+00:00,302.000
Benign4,2025-04-14 12:59:10.522000+00:00,2025-04-14 13:04:14.522000+00:00,304.000
Benign5,2025-04-14 13:08:32.522000+00:00,2025-04-14 13:11:07.522000+00:00,155.000
Benign6,2025-04-14 13:13:58.522000+00:00,2025-04-14 13:19:01.522000+00:00,303.000
DDOS_ICMP1,2025-04-03 15:20:04.868000+00:00,2025-04-03 15:21:50.869000+00:00,106.001
DDOS_ICMP2,2025-04-03 15:23:02.868000+00:00,2025-04-03 15:24:56.869000+00:00,114.001
DDOS_ICMP3,2025-04-03 15:26:03.868000+00:00,2025-04-03 15:27:56.879000+00:00,113.011


## Step 3 - Counter engineering (`dlBytes`, `ulBytes`)

Chronological first-order differences within `(scenario, ue_id)`. First observations, resets and large gaps become **NaN** - never clipped to zero.
Each delta covers `(previous timestamp_ms, timestamp_ms]`; both bounds are kept.

In [6]:
p.step3_counters()
cs = p.counter_stats
summary = pd.Series({
    "UE series": cs["n_series"], "first observations (NaN)": cs["n_first_observations"],
    "negative DL deltas (chronological)": cs["dl_resets"], "negative UL deltas (chronological)": cs["ul_resets"],
    "RNTI changes": cs["n_rnti_changes"], "resets coinciding with an RNTI change": cs["reset_and_rnti_change"],
    "resets WITHOUT an RNTI change": cs["reset_without_rnti_change"], "RNTI changes WITHOUT a reset": cs["rnti_change_without_reset"],
    "large gaps (> %.1f s)" % rp.GAP_THRESHOLD_S: cs["n_large_gaps"],
    "valid DL deltas": cs["dl_valid"], "invalid DL deltas": cs["dl_invalid"],
    "negative DL diffs if differenced in FILE order": p.stats["file_order_negatives"]["dl"],
}, name="value")
display(summary.to_frame())
print("dt between consecutive records (s):", cs["dt_describe"].round(4).to_dict())

,value
UE series,60
first observations (NaN),60
negative DL deltas (chronological),21
negative UL deltas (chronological),21
RNTI changes,21
resets coinciding with an RNTI change,21
resets WITHOUT an RNTI change,0
RNTI changes WITHOUT a reset,0
large gaps (> 1.5 s),0
valid DL deltas,7374


dt between consecutive records (s): {'count': 7395.0, 'mean': 1.0, 'std': 0.0013, 'min': 0.999, '25%': 1.0, '50%': 1.0, '75%': 1.0, 'max': 1.08}


In [7]:
# The 21 reset records: V2 stores NaN (the old clip(lower=0) would have stored 0 = fake 'zero traffic')
resets = p.cdf.loc[p.cdf["counter_reset"], ["scenario", "ue_id", "timestamp_ms", "rnti_prev", "rnti",
                                            "dlBytes_prev", "dlBytes", "dl_delta_raw", "dl_delta", "interval_start_ms", "interval_end_ms"]]
display(resets.head(8))
print("model-facing dl_delta on all reset rows is NaN:", bool(resets["dl_delta"].isna().all()))
print("RNTI-change x counter-reset cross-tab (rows):")
display(pd.crosstab(p.cdf["rnti_changed"], p.cdf["counter_reset"], rownames=["rnti_changed"], colnames=["counter_reset"]))

,scenario,ue_id,timestamp_ms,rnti_prev,rnti,dlBytes_prev,dlBytes,dl_delta_raw,dl_delta,interval_start_ms,interval_end_ms
2995,DDOS_ICMP1,148,1743693623868,082e,b00a,2627994.0,13487,-2614507.0,NaN,1.743694e+12,1.743694e+12
3010,DDOS_ICMP1,148,1743693638868,b00a,077c,889977.0,8029,-881948.0,NaN,1.743694e+12,1.743694e+12
3022,DDOS_ICMP1,148,1743693650868,077c,60e4,681262.0,0,-681262.0,NaN,1.743694e+12,1.743694e+12
3132,DDOS_ICMP1,149,1743693684868,dd04,523a,17701197.0,20211,-17680986.0,NaN,1.743694e+12,1.743694e+12
3173,DDOS_ICMP1,150,1743693666869,4a09,b191,814556.0,16329,-798227.0,NaN,1.743694e+12,1.743694e+12
3212,DDOS_ICMP1,150,1743693705869,b191,754a,2501657.0,342,-2501315.0,NaN,1.743694e+12,1.743694e+12
3215,DDOS_ICMP1,150,1743693708869,754a,cce0,52978.0,996,-51982.0,NaN,1.743694e+12,1.743694e+12
3774,DDOS_SYN1,131,1743691270932,e3a3,003a,11899163.0,0,-11899163.0,NaN,1.743691e+12,1.743691e+12


model-facing dl_delta on all reset rows is NaN: True
RNTI-change x counter-reset cross-tab (rows):


counter_reset,False,True
rnti_changed,,
False,7434,0
True,0,21


## Step 4 - Radio data quality (label-free flags; raw values preserved, no row removed)

* `q_rssi_lt_rsrp` - RSSI < RSRP (RSSI is normally >= RSRP); both columns untouched, no swap.
* `q_phr_gt_40`, `q_phr_gt_63` - PHR above the plausibility ceiling / outside a 6-bit index; **no offset applied**.
* `q_pusch_saturated` - `puschSnr == 63.5` (127 x 0.5, saturation code); excluded from `puschSnr_*` statistics.
* `q_all_zero_ul` - `pcmax, phr, pucchSnr, dlMcs, ulMcs` all exactly 0 on the same record; single zeros are not reinterpreted.

In [8]:
p.step4_quality()
q = p.q
flags = ["q_rssi_lt_rsrp", "q_phr_gt_40", "q_phr_gt_63", "q_pusch_saturated", "q_all_zero_ul", "oob_any"]
tot = pd.DataFrame({"records flagged": q[flags].sum(), "% of records": 100 * q[flags].mean(),
                    "scenarios affected": [(q.groupby("scenario")[f].sum() > 0).sum() for f in flags]})
display(tot.round(2))
print("rows before / after quality step:", len(p.dedup), "/", len(q), "(flags never remove rows)")
print("original *_flag_oob columns reproduce from the 02b ranges (mismatches per KPI):", {k: v for k, v in rp.recompute_oob(p.dedup).items() if v} or "none")

,records flagged,% of records,scenarios affected
q_rssi_lt_rsrp,2743,36.79,12
q_phr_gt_40,2657,35.64,12
q_phr_gt_63,38,0.51,10
q_pusch_saturated,132,1.77,12
q_all_zero_ul,86,1.15,3
oob_any,2657,35.64,12


rows before / after quality step: 7455 / 7455 (flags never remove rows)
original *_flag_oob columns reproduce from the 02b ranges (mismatches per KPI): none


In [9]:
print("PHR values above the ceiling (value: count):", dict(q.loc[q.phr > rp.PHR_PLAUSIBLE_MAX, "phr"].value_counts().sort_index(ascending=False).head(10)))
print("largest NON-saturated puschSnr:", q.loc[~q.q_pusch_saturated, "puschSnr"].max(), "| saturation code:", rp.PUSCH_SATURATION_CODE)
per = q.groupby("scenario")[flags].sum()
display(per[(per > 0).any(axis=1)])

PHR values above the ceiling (value: count): {68: np.int64(1), 67: np.int64(6), 66: np.int64(8), 65: np.int64(14), 64: np.int64(9), 63: np.int64(7), 62: np.int64(12), 61: np.int64(25), 60: np.int64(14), 59: np.int64(32)}
largest NON-saturated puschSnr: 56.0 | saturation code: 63.5


,q_rssi_lt_rsrp,q_phr_gt_40,q_phr_gt_63,q_pusch_saturated,q_all_zero_ul,oob_any
scenario,,,,,,
DDOS_ICMP1,242,212,0,37,30,212
DDOS_ICMP2,230,230,3,28,0,230
DDOS_ICMP3,228,228,8,16,0,228
DDOS_SYN1,212,211,1,13,1,211
DDOS_SYN2,222,222,3,3,0,222
DDOS_SYN3,228,228,6,4,0,228
DDOS_TCP1,212,212,1,1,0,212
DDOS_TCP2,228,228,5,7,0,228
DDOS_TCP3,232,232,2,4,0,232


## Step 5 - Per-second aggregation: one row per `(scenario, timestamp_s)`

In [10]:
p.step5_aggregate()
win = p.win
print(f"{len(win):,} windows x {win.shape[1]} columns | unique (scenario, timestamp_s): {not win.duplicated(['scenario','timestamp_s']).any()}")
print("reporting UEs per window:", win["n_ue_reporting"].value_counts().sort_index().to_dict())
nan_cols = win.isna().sum()
display(nan_cols[nan_cols > 0].rename("windows with NaN").to_frame())
display(win[["scenario", "timestamp_s", "n_records", "n_ue_reporting", "rsrq_mean", "rsrq_std", "rsrq_min", "rsrq_max", "rsrq_n",
             "puschSnr_mean", "puschSnr_n", "puschSnr_mean_nz", "dlBytes_delta_sum", "n_valid_dl_delta"]].head(5))

4,590 windows x 143 columns | unique (scenario, timestamp_s): True
reporting UEs per window: {1: 1794, 2: 2727, 3: 69}


,windows with NaN
puschSnr_mean,2
puschSnr_std,2
puschSnr_min,2
puschSnr_max,2
puschSnr_mean_nz,2
puschSnr_std_nz,2
puschSnr_min_nz,2
puschSnr_max_nz,2
dlBytes_delta_sum,36
ulBytes_delta_sum,36


,scenario,timestamp_s,n_records,n_ue_reporting,rsrq_mean,rsrq_std,rsrq_min,rsrq_max,rsrq_n,puschSnr_mean,puschSnr_n,puschSnr_mean_nz,dlBytes_delta_sum,n_valid_dl_delta
0,Benign1,1744634077,2,2,-11.25,0.75,-12.0,-10.5,2,19.25,2,19.25,NaN,0
1,Benign1,1744634078,2,2,-11.25,0.75,-12.0,-10.5,2,20.25,2,20.25,243820.0,2
2,Benign1,1744634079,2,2,-11.25,0.75,-12.0,-10.5,2,27.25,2,27.25,387113.0,2
3,Benign1,1744634080,2,2,-11.50,1.00,-12.5,-10.5,2,23.00,2,23.00,153731.0,2
4,Benign1,1744634081,2,2,-11.75,1.25,-13.0,-10.5,2,17.75,2,17.75,232002.0,2


## Step 6 - Feature metadata and the two model feature profiles

* **A - Conservative core:** RSRQ, DL MCS, UL MCS, DL BLER, UL BLER, PCMAX, PUCCH SNR, valid PUSCH SNR.
* **B - Extended candidate:** core + SINR + CQI (session-shift caution group).
* **Held out:** RSRP, RSSI, RI, PHR. **Sensitivity only:** byte-derived values and `_nz` variants. UE count, PMI and quality flags are never in a profile.

In [11]:
p.step6_metadata()
m = p.meta
prof = m["model_feature_profiles"]
print("Profile A (conservative core):", len(prof["A_conservative_core"]), "columns")
print("  KPIs:", m["primary_candidate_features"]["kpis"])
print("Profile B (extended candidate):", len(prof["B_extended_candidate"]), "columns; additional:", m["extended_candidate_features"]["additional_columns_over_core"])
print("  session-shift caution columns:", prof["session_shift_caution_features"])
print("Held out KPIs:", m["held_out_features"]["kpis"])
print("Sensitivity-only byte columns:", m["sensitivity_only_features"]["byte_derived"])
print("Section sizes:", {k: len(v["columns"]) for k, v in m.items() if isinstance(v, dict) and "columns" in v})

Profile A (conservative core): 32 columns
  KPIs: ['rsrq', 'dlMcs', 'ulMcs', 'dlBler', 'ulBler', 'pcmax', 'pucchSnr', 'puschSnr']
Profile B (extended candidate): 40 columns; additional: ['sinr_mean', 'sinr_std', 'sinr_min', 'sinr_max', 'cqi_mean', 'cqi_std', 'cqi_min', 'cqi_max']
  session-shift caution columns: ['sinr_mean', 'sinr_std', 'sinr_min', 'sinr_max', 'cqi_mean', 'cqi_std', 'cqi_min', 'cqi_max']
Held out KPIs: ['rsrp', 'rssi', 'ri', 'phr']
Sensitivity-only byte columns: ['dlBytes_delta_sum', 'ulBytes_delta_sum']
Section sizes: {'identifiers': 2, 'labels': 5, 'timestamps': 6, 'primary_candidate_features': 32, 'extended_candidate_features': 40, 'held_out_features': 16, 'sensitivity_only_features': 32, 'diagnostic_features': 22, 'excluded_features': 29, 'quality_flags': 20}


## Step 8 - Validation assertions (all hard checks must pass; audit targets are compared, not hard-coded)

In [12]:
p.step8_validate()
res = p.checks.frame()
res["result"] = np.where(res["passed"], "PASS", np.where(res["kind"] == "assert", "FAIL", "DISCREPANCY"))
display(res[["result", "name", "detail"]])
display(p.target_table)
print(f"hard assertions: {int((res.kind == 'assert').sum())} | failed: {len(p.checks.failed())} | audit-target discrepancies: {len(p.checks.discrepancies())}")
assert not p.checks.failed(), p.checks.failed()

,result,name,detail
0,PASS,schema: all required columns present,53 required columns present
1,PASS,schema: no unexpected extra columns (informati...,none
2,PASS,types: numeric columns are numeric dtype,non-numeric=[]
3,PASS,types: no +/-inf values in numeric columns,none
4,PASS,types: key/time columns are integer dtype,not integer=[]
...,...,...,...
58,PASS,audit target: rows_after_dedup,"expected 7,455, observed 7,455"
59,PASS,audit target: ue_series,"expected 60, observed 60"
60,PASS,audit target: dl_negative_deltas,"expected 21, observed 21"
61,PASS,audit target: ul_negative_deltas,"expected 21, observed 21"


,target,audit_expected,observed,status
0,input_rows,7490,7490,match
1,input_columns,53,53,match
2,scenarios,35,35,match
3,duplicate_extra_copies,35,35,match
4,rows_after_dedup,7455,7455,match
5,ue_series,60,60,match
6,dl_negative_deltas,21,21,match
7,ul_negative_deltas,21,21,match
8,radio_second_buckets,4590,4590,match


hard assertions: 53 | failed: 0 | audit-target discrepancies: 0


## Write the V2 outputs

In [13]:
written = p.write_outputs()
for f in written:
    print(f"{f.name:40s} {f.stat().st_size:>10,d} bytes")
post = p.checks.frame().tail(3)
display(post[["passed", "name", "detail"]])
assert not p.checks.failed(), p.checks.failed()
assert all(f.exists() and f.stat().st_size > 0 for f in written)

radio_window_features_v2.csv              2,887,639 bytes
radio_feature_columns_v2.json                16,368 bytes
radio_duplicate_removal_log_v2.csv            5,738 bytes
radio_counter_diagnostics_v2.csv          1,234,089 bytes
radio_quality_report_v2.md                   27,556 bytes


,passed,name,detail
63,True,"outputs: window CSV round-trips (shape, column...","4,590 rows x 143 columns"
64,True,outputs: feature-column JSON lists exactly the...,
65,True,outputs: input file untouched after writing (s...,


## Final verification: nothing pre-existing was modified; the run is reproducible

In [14]:
# 1) every pre-existing repository file is byte-identical to before the run
MANIFEST_AFTER = {str(x.relative_to(REPO)): sha(x) for x in PRE_EXISTING}
changed = [k for k in MANIFEST_BEFORE if MANIFEST_AFTER.get(k) != MANIFEST_BEFORE[k]]
missing = [k for k in MANIFEST_BEFORE if not (REPO / k).exists()]
print(f"{len(MANIFEST_BEFORE)} pre-existing files checked | changed: {len(changed)} | missing: {len(missing)}")
assert not changed and not missing
protected = ["Radio Dataset/02a-Radio analysis.ipynb", "Radio Dataset/02b-radio-cleaning.ipynb", "Radio Dataset/radio_clean.csv",
             "Radio Dataset/radio_feature_columns.json", "Multimodal Datasets/03-modality-alignment.ipynb",
             "Multimodal Datasets/multimodal_dataset.csv"]
assert all(k in MANIFEST_BEFORE for k in protected)
print("explicitly protected files present and unchanged:", len(protected))

# 2) a fresh, independent run reproduces the outputs byte for byte
with tempfile.TemporaryDirectory() as tmp:
    p2 = rp.RadioPipeline(INPUT, tmp, run_order_test=True).run()
    for name in [rp.OUT_WINDOW, rp.OUT_JSON, rp.OUT_DUPLOG, rp.OUT_COUNTERS, rp.OUT_REPORT]:
        same = sha(Path(tmp) / name) == sha(OUT / name)
        print(f"{name:40s} identical to the notebook's output: {same}")
        assert same, name

# 3) the exported column list equals the metadata
back = pd.read_csv(OUT / rp.OUT_WINDOW)
import json
meta = json.load(open(OUT / rp.OUT_JSON))
assert list(back.columns) == meta["exported_columns"]
print(f"window table reloaded: {back.shape[0]:,} rows x {back.shape[1]} columns; metadata matches exported columns")
print("\nALL STEPS COMPLETED SUCCESSFULLY")

17 pre-existing files checked | changed: 0 | missing: 0
explicitly protected files present and unchanged: 6


radio_window_features_v2.csv             identical to the notebook's output: True
radio_feature_columns_v2.json            identical to the notebook's output: True
radio_duplicate_removal_log_v2.csv       identical to the notebook's output: True
radio_counter_diagnostics_v2.csv         identical to the notebook's output: True
radio_quality_report_v2.md               identical to the notebook's output: True
window table reloaded: 4,590 rows x 143 columns; metadata matches exported columns

ALL STEPS COMPLETED SUCCESSFULLY
